# Rol 4 - Dinámica poblacional, ciclo completo y convergencia (Días 1, 2 y 4)

Este notebook comprueba desde un entorno limpio la configuración, la población inicial, la evaluación ordenada, el elitismo, las métricas de convergencia y la parada por estancamiento. Después ejecuta el ciclo completo con los operadores reales del Rol 3, la síntesis y el fitness del Rol 2.

La demostración del Día 2 usa una población pequeña para producir la primera corrida end-to-end reproducible; no constituye todavía el experimento de 12 configuraciones del Día 3.

In [ ]:
from dataclasses import asdict, replace
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import soundfile as sf
from IPython.display import SVG

RAIZ = Path.cwd().resolve()
if not (RAIZ / 'src' / 'algoritmo_genetico.py').is_file():
    RAIZ = RAIZ.parent
if not (RAIZ / 'src' / 'algoritmo_genetico.py').is_file():
    raise FileNotFoundError('Ejecuta desde la raíz o desde notebooks/.')
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.algoritmo_genetico import (
    calcular_estadisticas,
    detectar_estancamiento,
    ejecutar_ag,
    evaluar_poblacion,
    inicializar_poblacion,
    seleccionar_elites,
)
from src.configuracion import (
    ConfiguracionAG,
    SEMILLAS_EXPERIMENTALES,
    crear_plan_experimental,
)
from src.extraccion import cargar_melodia_csv
from src.fitness import evaluar_individuo
from src.operadores import seleccion_torneo
from src.sintesis import guardar_wav, sintetizar_instrumento

print(f'Raíz del proyecto: {RAIZ}')

## 1. Configuración acordada

La configuración reúne todos los hiperparámetros del Rol 4. Una validación temprana impide corridas ambiguas, por ejemplo, más élites que individuos o un torneo mayor que la población.

In [ ]:
config = ConfiguracionAG()
display(pd.DataFrame([asdict(config)]))

assert config.tam_poblacion == 50
assert config.elitismo == 2
assert config.tam_torneo == 3
assert config.max_generaciones == 50
assert config.paciencia_estancamiento == 10

## 2. Población y fitness reales del proyecto

Para mantener esta comprobación breve se evalúan cinco individuos nuevos. Se utilizan la melodía extraída por el Rol 1 y la referencia instrumental preparada por el Rol 2. Esto verifica la interfaz real, pero **no** es un experimento ni un resultado evolucionado.

In [ ]:
ruta_melodia = RAIZ / 'datos' / 'procesados' / 'melodia_01.csv'
ruta_referencia = RAIZ / 'datos' / 'audio' / 'referencias' / 'flauta_referencia_01.wav'

notas, duraciones = cargar_melodia_csv(ruta_melodia)
referencia, sr = sf.read(ruta_referencia, dtype='float32')
assert sr == 22050
assert referencia.ndim == 1

config_demo = replace(config, tam_poblacion=5, elitismo=2)
rng = np.random.default_rng(config_demo.semilla)
poblacion = inicializar_poblacion(config_demo, rng=rng)
poblacion_original = poblacion.copy()

def aptitud_real(individuo):
    return evaluar_individuo(
        individuo, notas, duraciones, referencia, metrica='stft'
    )

fitness = evaluar_poblacion(poblacion, aptitud_real)
np.testing.assert_array_equal(poblacion, poblacion_original)

display(pd.DataFrame({
    'individuo': np.arange(len(poblacion)),
    'fitness_stft': fitness,
}))

## 3. Elitismo, convergencia y conexión con Rol 3

El fitness se maximiza. En empates de elitismo se conserva el orden de la población; en el torneo se conserva el primer participante sorteado, según el contrato del Rol 3. Tanto los élites como el padre seleccionado son copias independientes.

In [ ]:
resumen = calcular_estadisticas(poblacion, fitness, generacion=0)
elites = seleccionar_elites(poblacion, fitness, config_demo.elitismo)
padre = seleccion_torneo(
    poblacion, fitness, k=config_demo.tam_torneo, rng=rng
)

assert elites.shape == (config_demo.elitismo, poblacion.shape[1])
assert not np.shares_memory(elites, poblacion)
assert not np.shares_memory(padre, poblacion)
assert np.isclose(resumen.fitness_maximo, fitness.max())

display(pd.DataFrame([asdict(resumen)]))
print('Padre seleccionado:', np.round(padre, 4))

## 4. Parada por estancamiento

Se compara la mejora acumulada del mejor global en una ventana de diez transiciones: `max(H[-10:]) - H[-11] < 1e-4`. Se necesitan once registros. No hay contador reiniciable: la ventana se desplaza cada generación; varias mejoras pequeñas pueden acumular una mejora suficiente para continuar.

In [ ]:
historial_estancado = [0.40] + [0.40001] * 10
historial_con_mejora = [0.40] * 10 + [0.4002]

assert detectar_estancamiento(
    historial_estancado,
    paciencia=config.paciencia_estancamiento,
    tolerancia=config.tolerancia_mejora,
)
assert not detectar_estancamiento(
    historial_con_mejora,
    paciencia=config.paciencia_estancamiento,
    tolerancia=config.tolerancia_mejora,
)
print('Criterio de parada verificado.')

## 5. Plan de 12 configuraciones

El diseño factorial cruza tamaño de población, elitismo y tasa de mutación. Las cinco semillas se aplicarán como réplicas en el Día 3. Mostrar el plan no equivale a ejecutar ni inventar mediciones.

In [ ]:
plan = crear_plan_experimental()
assert len(plan) == 12
assert len(SEMILLAS_EXPERIMENTALES) == 5

display(pd.DataFrame([
    {
        'configuracion': i + 1,
        'tam_poblacion': c.tam_poblacion,
        'elitismo': c.elitismo,
        'probabilidad_mutacion': c.probabilidad_mutacion,
        'probabilidad_cruce': c.probabilidad_cruce,
    }
    for i, c in enumerate(plan)
]))
print('Semillas de réplica:', SEMILLAS_EXPERIMENTALES)

## 6. Día 2: primera corrida end-to-end

Se ejecuta el ciclo completo con torneo, BLX-alpha y mutación gaussiana. La misma instancia de `Generator` gobierna inicialización y operadores; la síntesis conserva su semilla independiente. Se evalúan nuevamente las élites y se cuentan todas las llamadas reales al fitness.

In [ ]:
config_dia2 = replace(
    config,
    tam_poblacion=5,
    elitismo=1,
    max_generaciones=3,
    paciencia_estancamiento=10,
)
resultado_ag = ejecutar_ag(
    aptitud_real,
    config_dia2,
    seleccion='torneo',
    cruce='blend',
    mutacion='gaussiana',
)

historial = pd.DataFrame([
    asdict(fila) for fila in resultado_ag['historial']
])
assert resultado_ag['evaluaciones'] == (
    config_dia2.tam_poblacion
    * (resultado_ag['generaciones_ejecutadas'] + 1)
)
assert np.all(np.diff(resultado_ag['historial_mejor_fitness']) >= 0)
assert resultado_ag['poblacion_final'].shape == (
    config_dia2.tam_poblacion, 8
)

audio_mejor = sintetizar_instrumento(
    notas, duraciones, resultado_ag['mejor_individuo']
)
ruta_mejor = RAIZ / 'resultados' / 'audios' / 'rol4_dia2_mejor.wav'
guardar_wav(ruta_mejor, audio_mejor)

display(historial)
print('Mejor fitness:', resultado_ag['mejor_fitness'])
print('Evaluaciones reales:', resultado_ag['evaluaciones'])
print('Motivo de parada:', resultado_ag['motivo_parada'])
print('Audio evolucionado:', ruta_mejor)

## 7. Alcance verificado

Esta corrida conecta representación, melodía, síntesis, fitness, selección, cruce, mutación, elitismo, convergencia y exportación WAV. El historial queda listo para graficar curvas de convergencia. Las 12 configuraciones y sus réplicas corresponden al Día 3 y no se presentan aquí como resultados ejecutados.

## 8. Día 4: evidencia de convergencia disponible

Se reutilizan los seis checkpoints principales del experimento del Rol 3 (A, B y C; semillas 42 y 101). Esta celda valida y resume resultados ya guardados: **no ejecuta nuevas corridas ni llama al fitness**.

In [ ]:
carpeta_dia3 = RAIZ / 'resultados' / 'operadores' / 'dia3_2026-10-21_64617f8'
historial_dia3 = pd.read_csv(carpeta_dia3 / 'historial.csv')
resumen_dia3 = pd.read_csv(carpeta_dia3 / 'resumen.csv')

assert len(historial_dia3) == 306
assert set(historial_dia3['combinacion']) == {'A', 'B', 'C'}
assert set(historial_dia3['semilla']) == {42, 101}
assert (resumen_dia3['n'] == 2).all()

columnas = [
    'combinacion', 'n', 'media', 'desviacion_estandar',
    'diversidad_final_media',
]
display(resumen_dia3[columnas].rename(columns={
    'n': 'replicas',
    'media': 'mejor_fitness_medio',
    'desviacion_estandar': 'desviacion_estandar_muestral',
}))
display(SVG(filename=str(carpeta_dia3 / 'fitness.svg')))
display(SVG(filename=str(carpeta_dia3 / 'diversidad.svg')))

## 9. Pregunta teórica: convergencia prematura

La **convergencia prematura** ocurre cuando la población pierde diversidad demasiado pronto y se concentra alrededor de una solución subóptima. Puede detectarse combinando: (1) estancamiento del mejor fitness, (2) reducción de la dispersión del fitness y (3) caída sostenida de la diversidad genotípica; ninguna señal aislada basta para confirmarla.

En los resultados disponibles, A y B terminaron con diversidades medias 0.009409 y 0.009313: retuvieron aproximadamente 3.3 % de la diversidad inicial. C terminó en 0.125190 y retuvo aproximadamente 43.9 %. Esto evidencia una concentración mucho mayor en A/B, pero **no demuestra por sí solo convergencia prematura**: B obtuvo el mayor fitness medio final (0.730850) y las corridas usaron `tolerancia_mejora=0` para igualar el presupuesto, de modo que no probaron la parada operativa por estancamiento.

Para reducir el riesgo se puede disminuir la presión selectiva o el elitismo, aumentar o adaptar la mutación y, si el estancamiento persiste, introducir inmigrantes o reinicios parciales. Estas medidas son propuestas pendientes de experimentar; inmigración y reinicios no están implementados. La recomendación actual es B (torneo + uniforme + gaussiana), todavía preliminar porque solo hay dos réplicas y A ganó con la semilla 42 mientras B ganó con 101.

## 10. Actualización de integración: configuración del factorial

Después del cierre inicial del Rol 4, el Rol 5 integró el factorial de 12 configuraciones y cinco semillas para STFT y croma usando la combinación B. Esta celda consulta los resúmenes guardados: no ejecuta corridas, baseline ni fitness.

In [ ]:
carpeta_factorial = (
    RAIZ / 'resultados' / 'rol5' / 'dia3_factorial'
    / 'ejecucion_20261021T184758Z_c5dc431'
)
mejores_factorial = []
for metrica in ('stft', 'croma'):
    tabla = pd.read_csv(
        carpeta_factorial / metrica / 'resumen_por_configuracion.csv'
    )
    ag = tabla[tabla['metodo'] == 'algoritmo_genetico'].copy()
    assert len(ag) == 12
    assert (ag['n'] == 5).all()
    assert set(ag['combinacion_operadores']) == {'B'}
    mejor = ag.sort_values(
        ['media', 'desviacion_estandar'], ascending=[False, True]
    ).iloc[0]
    mejores_factorial.append({
        'metrica': metrica,
        'tam_poblacion': int(mejor['tam_poblacion']),
        'elitismo': int(mejor['elitismo']),
        'probabilidad_mutacion': mejor['probabilidad_mutacion'],
        'media': mejor['media'],
        'desviacion_estandar': mejor['desviacion_estandar'],
    })

display(pd.DataFrame(mejores_factorial))